# 04_03 Classification - NaiveBayes
Train and evaluate NaiveBayes.

[COMMAND_SO]
Command 1

[COMMAND_MUC_DICH]
- Muc tieu nghiep vu: Train NaiveBayes va visual ket qua de so sanh voi cac classifier khac.
- Muc tieu ky thuat: Hien thi bang metric va confusion matrix tren test set.

In [1]:
from pathlib import Path
import json
from pyspark.sql import SparkSession
from pyspark.ml.classification import NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
spark=(SparkSession.builder.appName('04_03_nb').master('local[2]').config('spark.sql.shuffle.partitions','16').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
PROJECT_ROOT=Path.cwd().resolve().parent if Path.cwd().name=='notebooks' else Path.cwd().resolve()
FEATURE_DIR=PROJECT_ROOT/'data'/'processed'/'features'
MODEL_DIR=PROJECT_ROOT/'models'/'classification'/'naive_bayes'
METRIC_DIR=PROJECT_ROOT/'reports'/'model_metrics'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)
train_df=spark.read.parquet(str(FEATURE_DIR/'classification_train')).select('order_id','label','features').dropna()
# --- Oversample minority class (class 1) to handle class imbalance ---
_c0 = train_df.filter(train_df['label'] == 0)
_c1 = train_df.filter(train_df['label'] == 1)
_n0, _n1 = _c0.count(), _c1.count()
print(f'Before oversampling: class_0={_n0}, class_1={_n1}, ratio={_n0/_n1:.2f}')
train_df = _c0.unionAll(_c1.sample(withReplacement=True, fraction=_n0/_n1, seed=42))
print(f'After oversampling: {train_df.count()} rows')
val_df=spark.read.parquet(str(FEATURE_DIR/'classification_val')).select('order_id','label','features').dropna()
test_df=spark.read.parquet(str(FEATURE_DIR/'classification_test')).select('order_id','label','features').dropna()
nb=NaiveBayes(featuresCol='features',labelCol='label',modelType='multinomial',smoothing=1.0)
m=nb.fit(train_df)
pred_val=m.transform(val_df)
pred_test=m.transform(test_df)
val_f1=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(pred_val)
val_acc=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='accuracy').evaluate(pred_val)
test_f1=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='f1').evaluate(pred_test)
test_acc=MulticlassClassificationEvaluator(labelCol='label',predictionCol='prediction',metricName='accuracy').evaluate(pred_test)
val_precision = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedPrecision').evaluate(pred_val)
val_recall    = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedRecall').evaluate(pred_val)
test_precision = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedPrecision').evaluate(pred_test)
test_recall    = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedRecall').evaluate(pred_test)
metrics={'model_family':'classification','model_name':'NaiveBayes','val_f1':float(val_f1),'val_accuracy':float(val_acc),'val_precision':float(val_precision),'val_recall':float(val_recall),'f1':float(test_f1),'accuracy':float(test_acc),'precision':float(test_precision),'recall':float(test_recall),'test_f1':float(test_f1),'test_accuracy':float(test_acc),'test_precision':float(test_precision),'test_recall':float(test_recall),'train_rows':train_df.count(),'val_rows':val_df.count(),'test_rows':test_df.count()}
print(metrics)
display(pd.DataFrame([metrics]))
cm_pdf=pred_test.groupBy('label','prediction').count().toPandas()
if not cm_pdf.empty:
    cm_table=cm_pdf.pivot(index='label', columns='prediction', values='count').fillna(0).sort_index().sort_index(axis=1)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm_table, annot=True, fmt='.0f', cmap='Blues')
    plt.title('Confusion Matrix - NaiveBayes (Test)')
    plt.xlabel('Prediction')
    plt.ylabel('Label')
    plt.tight_layout()
    plt.show()
m.write().overwrite().save(str(MODEL_DIR))
(METRIC_DIR/'classification_naive_bayes.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/06 00:23:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/04/06 00:23:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Before oversampling: class_0=52584, class_1=17025, ratio=3.09


After oversampling: 105187 rows


26/04/06 00:23:16 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/06 00:23:16 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


{'model_family': 'classification', 'model_name': 'NaiveBayes', 'val_f1': 0.8550965592506033, 'val_accuracy': 0.866519174041298, 'val_precision': 0.8564605568177333, 'val_recall': 0.8665191740412979, 'f1': 0.8607874099150912, 'accuracy': 0.8713462054170019, 'precision': 0.8603686965826592, 'recall': 0.8713462054170018, 'test_f1': 0.8607874099150912, 'test_accuracy': 0.8713462054170019, 'test_precision': 0.8603686965826592, 'test_recall': 0.8713462054170018, 'train_rows': 105187, 'val_rows': 14916, 'test_rows': 14916}


,model_family,model_name,val_f1,val_accuracy,val_precision,val_recall,f1,accuracy,precision,recall,test_f1,test_accuracy,test_precision,test_recall,train_rows,val_rows,test_rows
0,classification,NaiveBayes,0.855097,0.866519,0.856461,0.866519,0.860787,0.871346,0.860369,0.871346,0.860787,0.871346,0.860369,0.871346,105187,14916,14916


/var/folders/07/jh7lmpq57r72h5jdjy3vwy740000gn/T/ipykernel_91548/2367591904.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


557